# Task 4 — Notebook 01: Feature Engineering from CDL History

**Purpose (NAFSI Task 4):** Crop-type prediction (corn, soybean, third class) using **multi-year CDL** as features/labels, optionally **NDVI** and **SMAP**. This notebook loads the **full interim stacks** (CDL + optional NDVI/SMAP combined across years) so later cells can build the feature matrix and train/val splits (e.g. train on 2013–2022, validate on 2023 per config).

**Inputs (interim):**  
- `data/interim/cdl/cdl_stack_*.nc`  
- `data/interim/ndvi/ndvi_weekly_*.nc` (optional augmentation)  
- `data/interim/smap/smap_weekly_*.nc` (optional augmentation)  
- `data/processed/task2/rotation_metrics.parquet` — when Task 2 produces it, join rotation scores here

**Outputs:** feature matrix construction remains in following cells / notebooks; objects `cdl_stack`, `ndvi_all`, `smap_all` are ready in memory.

In [ ]:
# NOTE: This notebook was scaffolded with AI assistance.

import sys
from pathlib import Path

import pandas as pd
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root. cd to the repo and retry.")
sys.path.insert(0, str(REPO_ROOT))

from src.io.interim_loaders import (
    load_cdl_stack_from_interim,
    load_ndvi_weekly_all_years,
    load_smap_weekly_all_years,
)

with open(REPO_ROOT / "configs" / "task4_crop_mapping.yaml") as f:
    cfg = yaml.safe_load(f)

hist = cfg["cdl"]["history_years"]
test_y = int(cfg["cdl"]["test_year"])
print("CDL history years:", hist, "| held-out test year:", test_y)

In [ ]:
cdl_stack = load_cdl_stack_from_interim(REPO_ROOT)
ndvi_all = load_ndvi_weekly_all_years(REPO_ROOT)
smap_all = load_smap_weekly_all_years(REPO_ROOT)

print("cdl_stack", dict(cdl_stack.sizes))
print("ndvi_all", dict(ndvi_all.sizes))
print("smap_all", dict(smap_all.sizes))

rot_path = REPO_ROOT / "data" / "processed" / "task2" / "rotation_metrics.parquet"
if rot_path.is_file():
    rotation_scores = pd.read_parquet(rot_path)
    print("Loaded rotation metrics:", rotation_scores.shape)
else:
    rotation_scores = None
    print("No rotation_metrics.parquet yet — run Task 2 or skip rotation feature.")

# Next: encode sequences, transitions, 3×3 composition; labels from cdl_stack.sel(year=test_y).
